# Lecture 7 — Class Exercise
## Heatmap & Waterfall: Netflix Catalogue

> **Push to:** `week07/lecture07_exercise.ipynb`

**Rules:**
1. Heatmap: colour scale must match the data type (sequential for counts, diverging for above/below)
2. Waterfall: use green for additions, red for subtractions, blue for totals
3. Insight title tells the setup-conflict-resolution story (or at minimum states the finding)
4. Annotate at least one cell or bar directly

---


In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv('../data/netflix_catalogue.csv')
print(f"Loaded: {len(df)} titles")
print(df['type'].value_counts())
print(df.head())


Loaded: 3000 titles
type
Movie      1974
TV Show    1026
Name: count, dtype: int64
      type  release_year  added_year             genre        country rating  \
0    Movie          2014        2016  Sci-Fi & Fantasy         France  PG-13   
1    Movie          2010        2014     Documentaries  United States  TV-MA   
2  TV Show          2011        2012     Kids & Family  United States  TV-14   
3    Movie          2016        2018             Anime          India     PG   
4    Movie          2014        2016     Kids & Family         Canada  TV-MA   

   duration  
0       157  
1       127  
2         6  
3       134  
4        77  


In [2]:
print("Genres:", df['genre'].value_counts().head(8))
print("\nCountries:", df['country'].value_counts().head(8))
print("\nRatings:", df['rating'].value_counts())


Genres: genre
Sports                244
Sci-Fi & Fantasy      213
Kids & Family         209
Crime                 206
Drama                 204
Horror                199
Action & Adventure    198
Thrillers             195
Name: count, dtype: int64

Countries: country
United States     932
India             337
United Kingdom    261
Japan             187
France            176
Canada            164
South Korea       151
Mexico            138
Name: count, dtype: int64

Ratings: rating
TV-MA    840
TV-14    733
PG-13    589
R        312
PG       196
TV-PG    128
G         92
TV-Y7     57
TV-G      53
Name: count, dtype: int64


## Task 1 — Heatmap: content by rating and release decade

**What to build:** A heatmap showing the number of titles by **content rating** (y-axis) and **decade** (x-axis).

**Requirements:**
- Create a 'decade' column: `df['decade'] = (df['release_year'] // 10 * 10).astype(str) + 's'`
- Filter to TV-14, TV-MA, PG-13, R, PG only (most common ratings)
- Sequential colour scale (Blues)
- Values shown in cells (`text_auto=True`)
- Insight title about which rating dominates which decade


In [7]:
# Task 1
# YOUR CODE HERE
import pandas as pd
import plotly.express as px

# Create decade column
df['decade'] = (df['release_year'] // 10 * 10).astype(str) + 's'

# Keep only the required ratings
ratings_filter = ['TV-14', 'TV-MA', 'PG-13', 'R', 'PG']
filtered_df = df[df['rating'].isin(ratings_filter)]

# Create matrix
heatmap_data = pd.crosstab(
    filtered_df['rating'],
    filtered_df['decade']
)

# Order ratings logically
rating_order = ['PG', 'PG-13', 'R', 'TV-14', 'TV-MA']
heatmap_data = heatmap_data.reindex(rating_order)

# Order decades chronologically
heatmap_data = heatmap_data[sorted(heatmap_data.columns)]

# Build heatmap
fig = px.imshow(
    heatmap_data,
    text_auto=True,
    color_continuous_scale='Blues',
    aspect='auto',
    labels={
        "x": "Release Decade",
        "y": "Content Rating",
        "color": "Number of Titles"
    },
    title="Netflix Content Ratings Across Decades: Shift Toward Mature Audiences"
)

# Professional formatting
fig.update_layout(
    title={
        "x": 0.5,
        "xanchor": "center",
        "font": {"size": 22}
    },
    width=1000,
    height=550,
    font=dict(size=12),
    coloraxis_colorbar=dict(
        title="Titles"
    )
)

# Improve cell text readability
fig.update_traces(
    textfont={"size": 11}
)

fig.show()

## Task 2 — Waterfall: Movie vs TV Show additions by year

**What to build:** A waterfall chart showing how Netflix's **Movie library** grew year by year (2015-2022).

**Requirements:**
- Filter to Movies only
- Group by `added_year`, count titles per year
- Final bar should be the cumulative total
- Green bars (additions), blue total
- Annotation on the year with the largest single addition
- Insight title naming the growth story


In [ ]:
# Task 2
# YOUR CODE HERE
import pandas as pd
import plotly.graph_objects as go

# Filter Movies only
movies = df[df['type'] == 'Movie']

# Count movies added each year
yearly_additions = (
    movies.groupby('added_year')
    .size()
    .reset_index(name='count')
)

# Keep only 2015-2022
yearly_additions = yearly_additions[
    (yearly_additions['added_year'] >= 2015) &
    (yearly_additions['added_year'] <= 2022)
].sort_values('added_year')

# Find year with largest addition
max_year = yearly_additions.loc[yearly_additions['count'].idxmax(), 'added_year']
max_count = yearly_additions['count'].max()

# Waterfall chart
fig = go.Figure(go.Waterfall(
    name="Movies Added",
    orientation="v",
    measure=["relative"] * len(yearly_additions) + ["total"],
    x=[str(year) for year in yearly_additions['added_year']] + ["Total"],
    y=yearly_additions['count'].tolist() + [0],
    text=[str(x) for x in yearly_additions['count']] + [""],
    textposition="outside",
    increasing={"marker": {"color": "green"}},
    totals={"marker": {"color": "royalblue"}}
))

# Professional formatting
fig.update_layout(
    title={
        "text": "Netflix Movie Library Expansion Accelerated Rapidly Before Reaching a Massive Catalogue",
        "x": 0.5
    },
    xaxis_title="Year",
    yaxis_title="Movies Added",
    width=1000,
    height=600,
    template="plotly_white",
    showlegend=False
)

fig.show()